In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

In [ ]:
# Resolve A1 from the repository root, A1, notebooks, or another A1 subfolder.
for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "A1" / "src" / "DataLoader.py").is_file():
        PROJECT_DIR = base / "A1"
        break
    if (base / "src" / "DataLoader.py").is_file():
        PROJECT_DIR = base
        break
else:
    raise FileNotFoundError("Open the notebook from within the capstone repository.")

RAW_DATA_PATH = PROJECT_DIR / "data" / "processed" / "electricity_transformed.parquet"

df = pd.read_parquet(RAW_DATA_PATH, engine="pyarrow")
df_clean = df.copy() # Save a working copy

# Verify load
df.head()

In [ ]:
df.info()

## Network-stress target construction

The dataset does not contain direct engineering stress measurements such as line loading, reserve margin, frequency, voltage, outages, or constraint violations. The target below is therefore a **regional stress proxy** combining supply adequacy stress (high demand, high prices, and import dependence) with congestion/surplus stress (curtailment, exports, and high renewable penetration).

Thresholds are fitted on the first 70% of observations only. Intermediate severity and score series remain temporary; only the one-hour-ahead `stress_target_1h` column is added to the modelling dataframe.

In [ ]:
TRAIN_FRACTION = 0.70
STRESS_QUANTILES = [0.75, 0.90, 0.97]
STRESS_CLASSES = ["low", "medium", "high", "critical"]

# Preserve chronological order before fitting thresholds or shifting the target.
df_clean = df.sort_values("timestamp").reset_index(drop=True).copy()
train_end = int(len(df_clean) * TRAIN_FRACTION)
threshold_features = [
    "demand_gross",
    "price",
    "flow_imports",
    "curtailment",
    "flow_exports",
    "renewable_with_storage_proportion",
]

# Negative curtailment values are retained in the source data but represent
# no congestion stress when constructing this proxy.
training_for_thresholds = df_clean.iloc[:train_end][threshold_features].copy()
training_for_thresholds["curtailment"] = training_for_thresholds["curtailment"].clip(lower=0)
stress_thresholds = training_for_thresholds.quantile(STRESS_QUANTILES).T

def severity_points(series, feature):
    q75, q90, q97 = stress_thresholds.loc[feature, STRESS_QUANTILES]
    return pd.Series(
        np.select(
            [series >= q97, series >= q90, series >= q75],
            [3, 2, 1],
            default=0,
        ),
        index=series.index,
        dtype="int8",
    )

adequacy_score = (
    0.40 * severity_points(df_clean["demand_gross"], "demand_gross")
    + 0.35 * severity_points(df_clean["price"], "price")
    + 0.25 * severity_points(df_clean["flow_imports"], "flow_imports")
)

congestion_score = (
    0.60 * severity_points(df_clean["curtailment"].clip(lower=0), "curtailment")
    + 0.25 * severity_points(df_clean["flow_exports"], "flow_exports")
    + 0.15 * severity_points(
        df_clean["renewable_with_storage_proportion"],
        "renewable_with_storage_proportion",
    )
)

stress_score = np.maximum(adequacy_score, congestion_score)
current_stress_class = pd.Series(
    pd.cut(
        stress_score,
        bins=[-np.inf, 0.75, 1.50, 2.25, np.inf],
        labels=STRESS_CLASSES,
        right=False,
        ordered=True,
    ),
    index=df_clean.index,
)

# The row at time t receives the observed stress class at t + 1 hour.
df_clean["stress_target_1h"] = current_stress_class.shift(-1)
model_df = df_clean.dropna(subset=["stress_target_1h"]).copy()

In [ ]:
display(
    stress_thresholds.rename(
        columns={0.75: "elevated_75th", 0.90: "severe_90th", 0.97: "extreme_97th"}
    ).round(3)
)

class_counts = model_df["stress_target_1h"].value_counts(sort=False)
class_balance = pd.DataFrame(
    {
        "count": class_counts,
        "percentage": (class_counts / len(model_df) * 100).round(2),
    }
)
display(class_balance)

In [ ]:
added_columns = [column for column in df_clean.columns if column not in df.columns]
assert added_columns == ["stress_target_1h"]
assert df_clean["stress_target_1h"].isna().sum() == 1

print(f"Added column: {added_columns[0]}")
print(f"Modelling rows after removing the final unavailable target: {len(model_df):,}")
model_df[["timestamp", "stress_target_1h"]].head()